# 23 — PnL Sensitivity

How sensitive is PnL to: edge threshold, variance_rate parameter, position limits, and timing filters?

In [ ]:
import sys
sys.path.insert(0, '/Users/chriswang/Desktop/prediction-market-exploration/nba-edge')

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from nba_edge.data.s3_reader import discover_trade_dates, load_trades, load_boxscores_for_game, discover_game_ids
from nba_edge.data.ticker_parser import parse_ticker, is_game_winner
from nba_edge.data.game_alignment import match_ticker_to_game
from nba_edge.backtest.engine import BacktestEngine
from nba_edge.backtest.metrics import simulate_pnl
from nba_edge.models.analytical import AnalyticalWinProb

In [ ]:
# Load and run backtest (same setup as 21)
dates = discover_trade_dates(min_date='2026-04-21')
df = load_trades(dates)
game_tickers = [t for t in df['market_ticker'].unique().to_list() if is_game_winner(t)]

def run_all_backtests(variance_rate=0.44):
    engine = BacktestEngine(model=AnalyticalWinProb(variance_rate=variance_rate))
    results = []
    for ticker in sorted(game_tickers):
        parsed = parse_ticker(ticker)
        if not parsed.game_date:
            continue
        try:
            available = discover_game_ids(parsed.game_date)
        except Exception:
            continue
        matched = match_ticker_to_game(ticker, available)
        if not matched:
            continue
        snapshots = load_boxscores_for_game(matched, parsed.game_date)
        if len(snapshots) < 50 or snapshots[-1].get('game_status', 0) != 3:
            continue
        ticker_trades = df.filter(pl.col('market_ticker') == ticker)
        result = engine.run_game(ticker_trades, snapshots, ticker)
        if result and len(result.trades) > 0:
            results.append(result)
    return results

results_base = run_all_backtests(0.44)
print(f'Backtested {len(results_base)} markets')

In [ ]:
# Sensitivity 1: PnL vs edge threshold (fine-grained)
thresholds = np.arange(0.01, 0.25, 0.005)
pnl_by_thresh = []
trades_by_thresh = []

for t in thresholds:
    pnl = simulate_pnl(results_base, edge_threshold=t)
    pnl_by_thresh.append(pnl.gross_pnl_cents / 100)
    trades_by_thresh.append(pnl.n_trades_taken)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(thresholds * 100, pnl_by_thresh, linewidth=1.5)
axes[0].axhline(0, color='black', linestyle=':')
axes[0].set_ylabel('PnL ($)')
axes[0].set_title('PnL vs Edge Threshold')
axes[0].fill_between(thresholds * 100, pnl_by_thresh, 0, 
                     where=[p > 0 for p in pnl_by_thresh], alpha=0.1, color='green')
axes[0].fill_between(thresholds * 100, pnl_by_thresh, 0,
                     where=[p < 0 for p in pnl_by_thresh], alpha=0.1, color='red')

axes[1].plot(thresholds * 100, trades_by_thresh, linewidth=1.5, color='steelblue')
axes[1].set_xlabel('Edge Threshold (cents)')
axes[1].set_ylabel('Trades Taken')

plt.tight_layout()
plt.show()

In [ ]:
# Sensitivity 2: Variance rate parameter
var_rates = [0.30, 0.35, 0.40, 0.44, 0.50, 0.55, 0.60]
pnl_by_var = []

for vr in var_rates:
    results_vr = run_all_backtests(variance_rate=vr)
    pnl = simulate_pnl(results_vr, edge_threshold=0.05)
    pnl_by_var.append(pnl.gross_pnl_cents / 100)
    print(f'  var_rate={vr:.2f}: PnL=${pnl.gross_pnl_cents/100:.2f} ({pnl.n_trades_taken} trades)')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(var_rates, pnl_by_var, 'o-', linewidth=1.5)
ax.axhline(0, color='black', linestyle=':')
ax.axvline(0.44, color='red', linestyle='--', alpha=0.5, label='NBA default (0.44)')
ax.set_xlabel('Variance Rate (pts²/s)')
ax.set_ylabel('PnL ($) at 5% threshold')
ax.set_title('PnL Sensitivity to Variance Rate')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Sensitivity 3: Only trade in specific time windows
# Hypothesis: edges are larger in certain game phases
time_filters = [
    ('Full game', 0, 3000),
    ('1st half only', 1440, 3000),
    ('2nd half only', 0, 1440),
    ('4th quarter only', 0, 720),
    ('Last 5 min', 0, 300),
    ('Exclude last 2 min', 120, 3000),
]

print(f"{'Filter':25s} {'Trades':>8s} {'PnL ($)':>10s} {'Win Rate':>10s}")
print('-' * 58)

for name, min_secs, max_secs in time_filters:
    # Filter trades by seconds_remaining
    filtered_results = []
    for r in results_base:
        filtered_trades = [t for t in r.trades if min_secs <= t.seconds_remaining <= max_secs]
        if filtered_trades:
            from nba_edge.backtest.engine import GameBacktestResult
            filtered_r = GameBacktestResult(
                market_ticker=r.market_ticker, game_id=r.game_id,
                yes_team=r.yes_team, home_team=r.home_team, away_team=r.away_team,
                outcome_yes=r.outcome_yes, final_home_score=r.final_home_score,
                final_away_score=r.final_away_score, trades=filtered_trades,
            )
            filtered_results.append(filtered_r)
    
    pnl = simulate_pnl(filtered_results, edge_threshold=0.05)
    print(f'{name:25s} {pnl.n_trades_taken:>8d} {pnl.gross_pnl_cents/100:>10.2f} {pnl.win_rate:>10.1%}')

In [ ]:
# Sensitivity 4: Position sizing — cap max contracts per game
max_positions = [5, 10, 20, 50, 100, 500]

print(f"{'Max Position':>15s} {'PnL ($)':>10s} {'Trades':>8s}")
print('-' * 38)

for max_pos in max_positions:
    total_pnl = 0
    total_trades = 0
    for r in results_base:
        position = 0
        for t in r.trades:
            if abs(position) >= max_pos:
                continue
            if t.edge > 0.05:
                position += 1
                total_trades += 1
                if r.outcome_yes:
                    total_pnl += (100 - t.trade_price)
                else:
                    total_pnl -= t.trade_price
            elif t.edge < -0.05:
                position -= 1
                total_trades += 1
                if not r.outcome_yes:
                    total_pnl += t.trade_price
                else:
                    total_pnl -= (100 - t.trade_price)
    
    print(f'{max_pos:>15d} {total_pnl/100:>10.2f} {total_trades:>8d}')